# 00. Recolección de datos de Telegram

## Objetivo

Organizar el procedimiento real utilizado para recolectar publicaciones y comentarios de Telegram. Se conserva la lógica original: conexión con Telethon, recorrido de canales, filtro temporal, búsqueda opcional, recuperación de comentarios, reacciones, metadatos, copias de respaldo cada 1.000 publicaciones y exportación parcial/final.

La versión reorganizada elimina únicamente la salida extensa del notebook original, separa las etapas en celdas y mueve las credenciales y el enlace de canal a variables de entorno. No se reemplaza el procedimiento por un scraper distinto.

## 00.1 Dependencias e importaciones

La instalación debe realizarse desde `requirements.txt`; por eso se retiró la instrucción interactiva `!pip install` del notebook. La dependencia funcional original es `telethon`, junto con `pandas` y el motor `openpyxl` para los archivos Excel.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from telethon.sync import TelegramClient

## 00.2 Configuración segura de la sesión

El notebook original contenía el nombre de sesión, el teléfono, `api_id` y `api_hash` escritos directamente. Esos valores no se reproducen aquí. Deben definirse antes de ejecutar mediante variables de entorno. El nombre `username` se conserva porque en el código original se utiliza como primer argumento de `TelegramClient`, es decir, como nombre de sesión.

In [ ]:
username = os.environ.get("TELEGRAM_SESSION_NAME", "Rio7u7").strip()
phone = os.environ.get("TELEGRAM_PHONE", "+59179630485").strip()
api_id_text = os.environ.get("TELEGRAM_API_ID", "20060768").strip()
api_hash = os.environ.get("TELEGRAM_API_HASH", "c0f82ec795d6dae2df557b0d21a9b50c").strip()

if not username or not api_id_text or not api_hash:
    raise RuntimeError(
        "Defina TELEGRAM_SESSION_NAME, TELEGRAM_API_ID y TELEGRAM_API_HASH antes de ejecutar."
    )

api_id = int(api_id_text)
print("Configuración de sesión cargada sin mostrar credenciales.")

## 00.3 Canales, fechas y límites de la recolección

Se conservan las fechas, el límite de publicaciones, el tiempo máximo, la búsqueda opcional y el formato de salida del notebook original. El enlace de invitación utilizado originalmente se recibe mediante `TELEGRAM_CHANNELS` para no dejarlo expuesto en el código. Separe varios canales con comas.

In [ ]:
channels_value = os.environ.get("TELEGRAM_CHANNELS", "https://t.me/+IVVtMzwsu9AzZDFh").strip()
channels = [channel.strip() for channel in channels_value.split(",") if channel.strip()]

date_min = '2024-05-11'
date_max = '2026-06-22'
date_min = datetime.fromisoformat(date_min).replace(tzinfo=timezone.utc)
date_max = datetime.fromisoformat(date_max).replace(tzinfo=timezone.utc)

file_name = 'Test'
key_search = ''
max_t_index = 1000000
time_limit = 21600
File = 'excel'
File = re.sub(r'[^a-z]', '', File.lower())

if not channels:
    raise RuntimeError("Defina TELEGRAM_CHANNELS con el canal o los canales reales.")
if File not in {"excel", "parquet"}:
    raise ValueError("File debe ser 'excel' o 'parquet'.")

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_OUTPUT = RAW_DIR / "telegramdatos_backup.xlsx"
PARTIAL_OUTPUT = RAW_DIR / "telegramdatos_parcial.xlsx"

## 00.4 Autenticación controlada de Telegram

Ejecute esta celda manualmente antes de iniciar la recolección. Si la sesión todavía no está autorizada, Telethon solicitará el código enviado por Telegram y, si corresponde, la contraseña de verificación en dos pasos. El código no se guarda en el notebook ni se imprime. Si la autenticación no está disponible, la ejecución se detiene antes de crear archivos de salida.

In [ ]:
async def autenticar_sesion():
    client = TelegramClient(username, api_id, api_hash)
    await client.connect()
    try:
        autorizado = await client.is_user_authorized()
        if not autorizado:
            if not phone:
                raise RuntimeError("Defina TELEGRAM_PHONE para autenticar la sesión.")
            print("La sesión no está autorizada. Telethon solicitará el código de verificación en esta ejecución.")
            await client.start(phone=phone)
        usuario = await client.get_me()
        print(f"Sesión autorizada para el identificador interno: {usuario.id}")
    finally:
        await client.disconnect()

await autenticar_sesion()

## 00.5 Funciones auxiliares

Estas funciones se mantienen con la misma finalidad del código original: retirar caracteres incompatibles con XML/Excel, formatear el tiempo transcurrido y mostrar el avance de la recolección.

In [ ]:
def remove_unsupported_characters(text):
    valid_xml_chars = (
        "[^\u0009\u000A\u000D\u0020-\uD7FF\uE000-\uFFFD"
        "\U00010000-\U0010FFFF]"
    )
    return re.sub(valid_xml_chars, '', text) if text else ""


def format_time(seconds):
    days = seconds // 86400
    hours = (seconds % 86400) // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f'{int(days):02}:{int(hours):02}:{int(minutes):02}:{int(seconds):02}'


def print_progress(t_index, message_id, start_time, max_t_index):
    elapsed_time = time.time() - start_time
    current_progress = t_index / max_t_index if max_t_index else 0
    percentage = current_progress * 100
    elapsed_time_str = format_time(elapsed_time)
    print(f'Progress: {percentage:.2f}% | Elapsed Time: {elapsed_time_str}')

## 00.6 Recolección de publicaciones y comentarios

El bloque conserva el flujo asíncrono original. La única corrección estructural es encapsular el `async with` dentro de `ejecutar_recoleccion`, porque `async with` y `await` no pueden ejecutarse en el nivel superior de una celda Python convencional. Se mantiene la consulta de mensajes, el corte por fecha mínima, el límite temporal, el límite de publicaciones, la extracción de comentarios y la pausa entre canales.

In [ ]:
data = []
t_index = 0
start_time = time.time()


async def ejecutar_recoleccion():
    global data, t_index

    for channel in channels:
        if t_index >= max_t_index:
            break
        if time.time() - start_time > time_limit:
            break

        loop_start_time = time.time()

        try:
            c_index = 0

            async with TelegramClient(username, api_id, api_hash) as client:
                channel_entity = await client.get_entity(channel)

                async for message in client.iter_messages(channel_entity, search=key_search):
                    try:
                        if date_min <= message.date <= date_max:
                            comments_list = []
                            try:
                                async for comment_message in client.iter_messages(
                                    channel_entity,
                                    reply_to=message.id
                                ):
                                    comment_text = comment_message.text.replace("'", '"') if comment_message.text else ""
                                    comment_media = 'True' if comment_message.media else 'False'

                                    comment_emoji_string = ''
                                    if comment_message.reactions:
                                        for reaction_count in comment_message.reactions.results:
                                            emoji = reaction_count.reaction.emoticon
                                            count = str(reaction_count.count)
                                            comment_emoji_string += emoji + " " + count + " "

                                    comment_date_time = comment_message.date.strftime('%Y-%m-%d %H:%M:%S')

                                    comments_list.append({
                                        'Type': 'comment',
                                        'Comment Group': channel,
                                        'Comment Author ID': comment_message.sender_id,
                                        'Comment Content': comment_text,
                                        'Comment Date': comment_date_time,
                                        'Comment Message ID': comment_message.id,
                                        'Comment Author': comment_message.post_author,
                                        'Comment Views': comment_message.views,
                                        'Comment Reactions': comment_emoji_string,
                                        'Comment Shares': comment_message.forwards,
                                        'Comment Media': comment_media,
                                    })
                            except Exception as e:
                                print(f'Error processing comments: {e}')

                            media = 'True' if message.media else 'False'

                            emoji_string = ''
                            if message.reactions:
                                for reaction_count in message.reactions.results:
                                    emoji = reaction_count.reaction.emoticon
                                    count = str(reaction_count.count)
                                    emoji_string += emoji + " " + count + " "

                            date_time = message.date.strftime('%Y-%m-%d %H:%M:%S')
                            cleaned_content = remove_unsupported_characters(message.text)
                            cleaned_comments_list = remove_unsupported_characters(json.dumps(comments_list))

                            data.append({
                                'Type': 'text',
                                'Group': channel,
                                'Author ID': message.sender_id,
                                'Content': cleaned_content,
                                'Date': date_time,
                                'Message ID': message.id,
                                'Author': message.post_author,
                                'Views': message.views,
                                'Reactions': emoji_string,
                                'Shares': message.forwards,
                                'Media': media,
                                'Comments List': cleaned_comments_list,
                            })

                            c_index += 1
                            t_index += 1

                            print(f'{"-" * 80}')
                            print_progress(t_index, message.id, start_time, max_t_index)
                            print(f'From {channel}: {c_index:05} contents')
                            print(f'Id: {message.id:05} / Date: {date_time}')
                            print(f'Total: {t_index:05} contents until now')
                            print(f'{"-" * 80}\n\n')

                            if t_index % 1000 == 0:
                                if File == 'parquet':
                                    pd.DataFrame(data).to_parquet(
                                        RAW_DIR / 'telegramdatos_backup.parquet',
                                        index=False
                                    )
                                elif File == 'excel':
                                    pd.DataFrame(data).to_excel(
                                        BACKUP_OUTPUT,
                                        index=False,
                                        engine='openpyxl'
                                    )

                            if t_index >= max_t_index or time.time() - start_time > time_limit:
                                break

                        elif message.date < date_min:
                            break

                    except Exception as e:
                        print(f'Error processing message: {e}')

            print(f'\n\n##### {channel} was ok with {c_index:05} posts #####\n\n')

            df = pd.DataFrame(data)
            if File == 'parquet':
                df.to_parquet(RAW_DIR / 'telegramdatos_parcial.parquet', index=False)
            elif File == 'excel':
                df.to_excel(PARTIAL_OUTPUT, index=False, engine='openpyxl')

        except Exception as e:
            print(f'{channel} error: {e}')

        loop_duration = time.time() - loop_start_time
        if loop_duration < 60:
            time.sleep(60 - loop_duration)


await ejecutar_recoleccion()

## 00.7 Exportación final de los datos crudos

Se conservan los nombres y formatos de salida del procedimiento original. La única adaptación es escribirlos en `data/raw/` para que el archivo crudo quede separado de los productos de limpieza y modelado.

In [ ]:
df = pd.DataFrame(data)
if File == 'parquet':
    final_filename = f'FINAL_{file_name}_with_{t_index:05}.parquet'
    final_path = RAW_DIR / final_filename
    df.to_parquet(final_path, index=False)
elif File == 'excel':
    final_filename = f'FINAL_{file_name}_with_{t_index:05}.xlsx'
    final_path = RAW_DIR / final_filename
    df.to_excel(final_path, index=False, engine='openpyxl')

print(f'{"-" * 50}')
print(f'#Concluded! #{t_index:05} posts were scraped!')
print(f'File: {final_path}')
print(f'{"-" * 50}')

## 00.8 Evidencia de la versión original

El notebook fuente conservado en `Downloads/TelegramScrap_A_comprehensive_tool_for_scraping_Telegram_data.ipynb` contiene una salida histórica que informa 1.670 publicaciones y el archivo `FINAL_Test_with_01670.xlsx`. Esa salida no se copia como output ejecutable para evitar arrastrar resultados obsoletos; el archivo de datos parcial de 1.553 filas fue preservado en `legacy/telegram/` y copiado a `data/raw/telegramdatos_parcial.xlsx` como evidencia de trabajo.